# 138 — Aprendizaje por imitación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=138)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Supervivencia p^T

- (a) p=0.95: T=10 → 0.599; T=20 → 0.358; T=50 → 0.077.
- (b) p=0.99: T=10 → 0.904; T=20 → 0.818; T=50 → 0.605.
- (c) `p = 0.9^(1/100) ≈ 0.99895`: haría falta un 99.9 % de acierto por paso.
Reducir ε exige perfección exponencialmente creciente con T; cambiar la
*distribución* de entrenamiento (DAgger) ataca el término T² directamente y
por eso escala mejor que perseguir el último decimal de accuracy.


In [ ]:
for p in (0.95, 0.99):
    print(p, [round(p**T, 3) for T in (10, 20, 50)])
print("p necesario T=100:", round(0.9 ** (1/100), 5))


## Solución 2 — Simulación BC vs DAgger

Resultado típico con seed 138: BC ≈ 30-40 % de episodios terminan en la celda
5; post-DAgger ≈ 85-95 %. El mecanismo es visible en la simulación: con BC, el
primer error lleva a la celda 4 o 6, donde la acción aleatoria tiene 2/3 de
probabilidad de no volver — y la deriva continúa. Con las celdas 4-6
etiquetadas, un desvío de un paso se corrige con prob. 0.95 y solo los dobles
errores consecutivos escapan de la zona cubierta.


In [ ]:
import random
random.seed(138)

def episodio(cobertura):
    celda = 5
    for _ in range(20):
        if celda in cobertura and random.random() < 0.95:
            celda += (5 > celda) - (5 < celda)  # hacia 5 (0 si ya esta)
        else:
            celda += random.choice((-1, 0, 1))
        celda = max(0, min(9, celda))
    return celda == 5

for nombre, cob in (("BC", {5}), ("DAgger", {4, 5, 6})):
    exitos = sum(episodio(cob) for _ in range(2000))
    print(f"{nombre}: {exitos/2000:.1%}")


## Solución 3 — Qué corrige cada intervención

- (a) Más demos de la misma ruta: reduce **ε** en distribución; no toca el
  shift.
- (b) DAgger: corrige el **covariate shift** (y de paso algo de ε en los
  estados nuevos).
- (c) Red mayor: reduce **ε** (si había subajuste); el shift permanece.
- (d) Ruido en los estados: ataca *parcialmente* el **shift** — amplía la
  distribución de entrenamiento hacia estados vecinos sin experto
  interactivo; es la versión barata de DAgger y la razón de su popularidad.
- (e) Recortar T: no corrige nada, pero **reduce el daño** (el T² actúa sobre
  un horizonte menor) — es mitigación, no cura.


## Solución 4 — Presupuesto de etiquetado

Para T=50, la opción B es la defendible: el término dominante del fracaso de
BC es el T² del covariate shift, y ninguna cantidad de demos en-ruta lo
elimina; 600 etiquetas colocadas en los estados que la política realmente
visita compran cobertura donde ocurren los fallos. Confirmación práctica:
(1) la accuracy de validación en D será *similar* en A y B, pero (2) la tasa
de éxito por episodio será claramente superior en B, y (3) el histograma de
estados visitados por la política B se mantendrá concentrado cerca de la ruta
(sin colas de deriva). Si A y B rindieran igual, la tarea tenía horizonte
efectivo corto o el experto inicial ya cubría los estados de error.
